# Prompts and LCEL with LangChain

This notebook shows three ways to send instructions to a chat model: a one-off message list, a reusable prompt template, and an LCEL chain that connects prompt, model, and output processing.

The examples use `OPENAI_API_KEY` from a local `.env` file. Keep API keys out of source control.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-6-luna",
)

## 1. Send a One-Off Prompt

A message list is useful for a single request. Each message has a role and content; the system message sets the assistant's style, while the user message contains the task.

In [3]:
# Chat models accept raw lists of role-content dictionaries directly.
# Valid standard roles are "system", "user", and "assistant".
prompt = [
    {"role": "system", "content": "You are a senior technical writer."},
    {"role": "user", "content": "Explain the difference between concurrency and parallelism in one sentence."}
]

res = llm.invoke(prompt)
print(res.content)

Concurrency is managing multiple tasks that make progress over overlapping periods, while parallelism is executing multiple tasks at the same time.


## 2. Reuse a Prompt Template

Use a template when the instruction stays the same but values like the target language or text change between requests. The `{language}` and `{query}` placeholders are filled in when you format or invoke the template.

In [4]:
# ChatPromptTemplate lets the same messages be reused with different inputs.
from langchain_core.prompts import ChatPromptTemplate

In [5]:
# Keep the instructions fixed and expose only the values that change per request.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a professional localization specialist. Translate the text into {language}."),
        ("user", "{query}")
    ]
)

# Inspect the template before filling its placeholders.
print(prompt)

input_variables=['language', 'query'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='You are a professional localization specialist. Translate the text into {language}.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})]


In [6]:
# format_messages returns role-tagged message objects ready for the chat model.
final_prompt = prompt.format_messages(
    language="German",
    query="The database connection timed out."
)
print(final_prompt)

# Send the formatted messages to the model.
res = llm.invoke(final_prompt)
print(res.content)

[SystemMessage(content='You are a professional localization specialist. Translate the text into German.', additional_kwargs={}, response_metadata={}), HumanMessage(content='The database connection timed out.', additional_kwargs={}, response_metadata={})]
Bei der Verbindung zur Datenbank ist eine Zeitüberschreitung aufgetreten.


In [7]:
# prompt.invoke returns a PromptValue that can be inspected before the model call.
prompt_value = prompt.invoke({
    "language": "German",
    "query": "The database connection timed out."
})
print(prompt_value)

# Reuse the formatted value rather than formatting the same inputs again.
res = llm.invoke(prompt_value)
print(res.content)

messages=[SystemMessage(content='You are a professional localization specialist. Translate the text into German.', additional_kwargs={}, response_metadata={}), HumanMessage(content='The database connection timed out.', additional_kwargs={}, response_metadata={})]
Bei der Datenbankverbindung ist eine Zeitüberschreitung aufgetreten.


## 3. Connect the Steps with LCEL

LangChain Expression Language (LCEL) uses `|` to pass each step's output to the next. The chain below reuses the same prompt and model without manually formatting messages or calling each step separately.

In [8]:
# The prompt fills its placeholders, then LCEL passes the result to the model.
chain = prompt | llm
res = chain.invoke({"language": "Japanese", "query": "The deployment was successful."})
print(res.content)

デプロイは成功しました。


### Return Plain Text from the Chain

A chat model normally returns an `AIMessage`, which includes metadata as well as text. `StrOutputParser` extracts just the response text, which is convenient when the next step only needs a string.

In [9]:
# StrOutputParser converts the model's AIMessage into just its text content.
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
chain = prompt | llm | output_parser
res = chain.invoke({"language": "Japanese", "query": "The deployment was successful."})
print(res)

デプロイに成功しました。


### Add a Small Python Step

Once the parser returns a string, another function can transform it before the chain returns its result. The next example uppercases the model response to show that ordinary Python functions can be part of an LCEL pipeline.

In [10]:
# The parser outputs a string, so this function can transform the text directly.
def to_upper(result: str) -> str:
    return result.upper()

# Add the Python transformation as the final step in the pipeline.
chain = prompt | llm | output_parser | to_upper
res = chain.invoke({"language": "English", "query": "デプロイは成功しました。"})
print(res)

DEPLOYMENT SUCCEEDED.
